In [2]:
import json
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def stream_arxiv(file_path, limit=None):
    papers = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            
            try:
                data = json.loads(line)
            except json.JSONDecodeError:
                continue
            
            papers.append({
                "id": data.get("id"),
                "title": data.get("title"),
                "abstract": data.get("abstract"),
                "categories": data.get("categories"),
                "authors": data.get("authors"),
                "update_date": data.get("update_date")
            })
    return papers

papers = stream_arxiv("arxiv-metadata-oai-snapshot.json", limit=10000)
print(f"Loaded {len(papers)} papers from the full dataset.")

Loaded 10000 papers from the full dataset.


In [3]:
def clean_text(text):
    text = (text or "").lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text

for p in papers:
    title = p.get("title") or ""
    abstract = p.get("abstract") or ""
    p["text"] = clean_text(title + " " + abstract)

In [4]:
corpus = [p["text"] for p in papers]

vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
tfidf_matrix = vectorizer.fit_transform(corpus)

In [5]:
def build_user_profile(user_indices):
    profile = tfidf_matrix[user_indices].mean(axis=0)
    return np.asarray(profile)

In [6]:
def recommend_content(user_profile, top_k=10):
    scores = cosine_similarity(user_profile, tfidf_matrix).flatten()
    top_indices = scores.argsort()[-top_k:][::-1]
    return top_indices, scores

In [7]:
def normalize(x):
    x = np.asarray(x, dtype=float)
    spread = x.max() - x.min()
    if spread == 0:
        return np.zeros_like(x)
    return (x - x.min()) / spread

# Popularity from interaction frequency (synthetic demo logs, deterministic)
rng = np.random.default_rng(42)
simulated_interactions = rng.integers(0, len(papers), size=min(50000, max(5000, len(papers) // 2)))
popularity = np.bincount(simulated_interactions, minlength=len(papers)).astype(float)

In [8]:
def hybrid_recommend(user_indices, top_k=10):
    user_profile = build_user_profile(user_indices)
    content_scores = cosine_similarity(user_profile, tfidf_matrix).flatten()
    content_scores = normalize(content_scores)
    
    pop_scores = normalize(popularity)
    
    final_scores = 0.7 * content_scores + 0.3 * pop_scores
    final_scores = normalize(final_scores)
    
    # Avoid recommending already interacted papers
    final_scores[user_indices] = -1.0
    
    top_indices = final_scores.argsort()[-top_k:][::-1]
    score_components = {
        "content": content_scores,
        "popularity": pop_scores,
        "final": final_scores
    }
    return top_indices, score_components

In [9]:
def explain_recommendation(paper_idx, user_indices, score_components):
    feature_names = vectorizer.get_feature_names_out()
    
    user_profile = build_user_profile(user_indices)
    paper_vector = tfidf_matrix[paper_idx]
    
    # Overlapping keywords between user profile and candidate paper
    overlap = paper_vector.multiply(user_profile).toarray().flatten()
    ranked = overlap.argsort()[::-1]
    top_features = [i for i in ranked if overlap[i] > 0][:5]
    keywords = [feature_names[i] for i in top_features] if top_features else ["general relevance"]
    
    content_score = float(score_components["content"][paper_idx])
    pop_score = float(score_components["popularity"][paper_idx])
    
    weighted_content = 0.7 * content_score
    weighted_popularity = 0.3 * pop_score
    total = weighted_content + weighted_popularity + 1e-12
    
    content_pct = 100 * weighted_content / total
    pop_pct = 100 * weighted_popularity / total
    
    return {
        "keywords": keywords,
        "content_contribution_pct": round(content_pct, 2),
        "popularity_contribution_pct": round(pop_pct, 2),
        "message": (
            f"Recommended because of shared topics: {', '.join(keywords)}. "
            f"Score contribution -> content: {content_pct:.1f}%, popularity: {pop_pct:.1f}%."
        )
    }

In [10]:
user_history = [1, 10, 25, 40]  # papers user interacted with

results, score_components = hybrid_recommend(user_history)

for idx in results:
    print("\nTitle:", papers[idx]["title"][:180])
    print("Relevance score (0-1):", round(float(score_components["final"][idx]), 4))
    explanation = explain_recommendation(idx, user_history, score_components)
    print("Explanation:", explanation["message"])


Title: Placeholder Substructures III: A Bit-String-Driven ''Recipe Theory'' for
  Infinite-Dimensional Zero-Divisor Spaces
Relevance score (0-1): 0.8832
Explanation: Recommended because of shared topics: zds, boxkites, genus, therebyscalefree networks, simple bitmanipulation. Score contribution -> content: 100.0%, popularity: 0.0%.

Title: Maximal probabilities of convolution powers of discrete uniform
  distributions
Relevance score (0-1): 0.4049
Explanation: Recommended because of shared topics: constant. Score contribution -> content: 1.2%, popularity: 98.8%.

Title: Cosmological Simulations of the Preheating Scenario for Galaxy Cluster
  Formation: Comparison to Analytic Models and Observations
Relevance score (0-1): 0.3623
Explanation: Recommended because of shared topics: previous, smooth, groups, results, work. Score contribution -> content: 8.0%, popularity: 92.0%.

Title: Large Scale Intermittency in the Atmospheric Boundary Layer
Relevance score (0-1): 0.3518
Explanation: Re

In [14]:
def _apk(actual, predicted, k=10):
    if k <= 0:
        return 0.0
    predicted = predicted[:k]
    score = 0.0
    hits = 0.0
    seen = set()

    for i, p in enumerate(predicted, start=1):
        if p in actual and p not in seen:
            hits += 1.0
            score += hits / i
            seen.add(p)

    return score / min(len(actual), k) if actual else 0.0

def _primary_category(cat_str):
    if not cat_str:
        return None
    # arXiv categories are space-separated; use the first one as primary topic
    return cat_str.split()[0]

def evaluate_hybrid_model(n_users=200, history_size=5, top_k=10, seed=123):
    if len(papers) < history_size + 1:
        raise ValueError("Not enough papers to run evaluation.")

    rng = np.random.default_rng(seed)

    # Build topic buckets so synthetic users have coherent interests
    category_to_indices = {}
    for i, paper in enumerate(papers):
        cat = _primary_category(paper.get("categories"))
        if cat is None:
            continue
        category_to_indices.setdefault(cat, []).append(i)

    valid_categories = [
        cat for cat, idxs in category_to_indices.items()
        if len(idxs) >= history_size + 1
    ]
    if not valid_categories:
        raise ValueError("No categories have enough papers for evaluation.")

    precisions = []
    recalls = []
    maps = []
    hit_count = 0
    recommended_items = set()

    for _ in range(n_users):
        chosen_cat = valid_categories[rng.integers(0, len(valid_categories))]
        pool = np.array(category_to_indices[chosen_cat], dtype=int)

        sampled = rng.choice(pool, size=history_size + 1, replace=False)
        train_history = sampled[:-1].tolist()
        holdout = int(sampled[-1])

        recs, _ = hybrid_recommend(train_history, top_k=top_k)
        recs_list = recs.tolist()
        recommended_items.update(recs_list)

        relevant = {holdout}
        hits = len(relevant.intersection(recs_list))
        hit_count += 1 if hits > 0 else 0

        precisions.append(hits / top_k)
        recalls.append(float(hits))
        maps.append(_apk(relevant, recs_list, k=top_k))

    results = {
        "users_evaluated": n_users,
        f"precision@{top_k}": float(np.mean(precisions)),
        f"recall@{top_k}": float(np.mean(recalls)),
        f"map@{top_k}": float(np.mean(maps)),
        f"hit_rate@{top_k}": float(hit_count / n_users),
        "catalog_coverage": float(len(recommended_items) / len(papers)),
        "evaluation_protocol": "category-aware leave-one-out"
    }
    return results

metrics = evaluate_hybrid_model(n_users=200, history_size=5, top_k=10)
print("Evaluation metrics:")
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

Evaluation metrics:
users_evaluated: 200
precision@10: 0.0085
recall@10: 0.0850
map@10: 0.0310
hit_rate@10: 0.0850
catalog_coverage: 0.0567
evaluation_protocol: category-aware leave-one-out
